In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
HERE = %pwd
sys.path.append(os.path.dirname(HERE))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
    
import numpy as np
import pandas as pd
import copy
import pickle
import time
import collections
from tqdm import tqdm
from collections import defaultdict

In [2]:
from src import utils
rng = utils.set_seed()

dir_parent = utils.dir_parent
version_exp = utils.version_exp
dir_workspace = f"{dir_parent}/research/TFCSR"

In [3]:
data_names = ["MovieLens", "Job"] + [f"ARD_{a}" for a in [
    "CDs_and_Vinyl", "Movies_and_TV", "Toys_and_Games", "Sports_and_Outdoors"
]]
N_icl = [1]
flag_replace_NER = False

dd = defaultdict(dict)
for data_name in tqdm(data_names):
    from src.data_loader import Loader
    loader = Loader(dir_workspace, version_exp, data_name, N_icl=N_icl, flag_replace_NER=flag_replace_NER)
    dict_data = loader.load_data()

    dd_text_profile = dict_data["profile"]
    if len(dd_text_profile) > 0:
        for a,d in dd_text_profile.items():
            s = pd.Series({k : utils.compute_token(v) for k,v in d.items()})
            dd["profile"][a] = f"${s.mean():.1f} \\pm {s.std():.1f}$"
        
    d = dict_data["items"]["history"]
    s = pd.Series({k : utils.compute_token(v) for k,v in d.items()})
    dd["1"][data_name] = f"${s.mean():.1f} \\pm {s.std():.1f}$"

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:11<00:00,  1.96s/it]


In [4]:
df = pd.DataFrame(pd.concat([pd.Series(dd["profile"]), pd.Series(dd["1"])], axis=0)).T
display(df)

print(df.to_latex(escape=False))

,profile,profile_mid-career,profile_new-graduate,MovieLens,Job,ARD_CDs_and_Vinyl,ARD_Movies_and_TV,ARD_Toys_and_Games,ARD_Sports_and_Outdoors
0,$22.2 \pm 1.5$,$105.2 \pm 18.2$,$61.1 \pm 3.0$,$15.0 \pm 3.2$,$359.1 \pm 206.3$,$129.7 \pm 77.9$,$147.7 \pm 73.6$,$154.5 \pm 73.2$,$136.2 \pm 75.6$


\begin{tabular}{llllllllll}
\toprule
 & profile & profile_mid-career & profile_new-graduate & MovieLens & Job & ARD_CDs_and_Vinyl & ARD_Movies_and_TV & ARD_Toys_and_Games & ARD_Sports_and_Outdoors \\
\midrule
0 & $22.2 \pm 1.5$ & $105.2 \pm 18.2$ & $61.1 \pm 3.0$ & $15.0 \pm 3.2$ & $359.1 \pm 206.3$ & $129.7 \pm 77.9$ & $147.7 \pm 73.6$ & $154.5 \pm 73.2$ & $136.2 \pm 75.6$ \\
\bottomrule
\end{tabular}

